# Fraud Detection EDA + DVC Assessment for `haminh109/Fraud-Detection`

## Objective
Notebook này thực hiện **EDA thực chiến cho fraud detection** và sau đó đánh giá **có nên dùng DVC cho repository hiện tại hay không**.

### Mục tiêu trong notebook
1. Kiểm tra dataset kỹ, bám sát bài toán fraud detection  
2. Tập trung vào các điểm quan trọng cho modeling:
   - class imbalance
   - fraud ratio
   - missing data
   - duplicates
   - outliers
   - phân phối feature
   - sự khác biệt giữa fraud và non-fraud
   - tương quan giữa các biến
   - time pattern nếu có  
3. Liên hệ insight EDA với các metric nhóm đang quan tâm:
   - **Recall**
   - **Precision**
   - **F1-score**
   - **PR-AUC**
4. Đánh giá DVC theo hướng **repo-specific**, không nói chung chung

## Repository-specific assumptions
- Repo hiện tại được giả định có các file:
  - `raw_data/train_transaction.csv.gz`
  - `raw_data/train_identity.csv`
  - `raw_data/test_transaction.csv.gz`
  - `raw_data/test_identity.csv`
  - `raw_data/sample_submission.csv`
- Vì các file CSV lớn thường không preview được đầy đủ trên GitHub, notebook này sẽ **tự kiểm tra schema khi chạy**, thay vì hard-code toàn bộ header.
- Bảng EDA chính trong notebook là:
  - `train_transaction` làm bảng gốc
  - nếu có `train_identity`, notebook sẽ **left join theo `TransactionID`** để mở rộng feature
- Nếu target không được detect tự động, hãy chỉnh ở cell config.
- Nếu RAM máy hạn chế, đặt:
  - `MERGE_IDENTITY = False`, hoặc
  - `NROWS = 100_000` để chạy thử nhanh

## Scope
- Chưa tách thành **Data Quality Checks module** riêng
- Chưa tách thành **Feature Engineering module** riêng
- Nhưng EDA vẫn sẽ chỉ ra các tín hiệu quan trọng cho bước modeling sau này

In [ ]:
import gc
import math
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import Markdown, display

try:
    from scipy.stats import ks_2samp
except Exception:
    ks_2samp = None

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)

# =========================
# User-configurable settings
# =========================
RANDOM_STATE = 42
NROWS = None                    # ví dụ: 100_000 để dry-run nhanh
LOAD_IDENTITY = True
MERGE_IDENTITY = True
SAVE_FIGURES = False

PLOT_SAMPLE_SIZE = 120_000
BALANCED_CLASS_SAMPLE_PER_CLASS = 10_000
MAX_NUMERIC_PLOTS = 8
MAX_COMPARE_PLOTS = 8
MAX_CORR_FEATURES = 30
MAX_CATEGORICAL_FEATURES = 5
MISSINGNESS_ALERT_THRESHOLD = 0.90
CATEGORY_MAX_LEVELS_FOR_PLOT = 12

MANUAL_TARGET_COL = None        # ví dụ: "isFraud" nếu muốn set thủ công
MANUAL_ID_COL = None            # ví dụ: "TransactionID" nếu muốn set thủ công

def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "raw_data").exists():
            return candidate
    return start

REPO_ROOT = find_repo_root()
RAW_DATA_DIR = REPO_ROOT / "raw_data"
FIG_DIR = REPO_ROOT / "reports" / "figures" / "eda"

TRAIN_TRANSACTION_PATH = RAW_DATA_DIR / "train_transaction.csv.gz"
TRAIN_IDENTITY_PATH = RAW_DATA_DIR / "train_identity.csv"
TEST_TRANSACTION_PATH = RAW_DATA_DIR / "test_transaction.csv.gz"
TEST_IDENTITY_PATH = RAW_DATA_DIR / "test_identity.csv"
SAMPLE_SUBMISSION_PATH = RAW_DATA_DIR / "sample_submission.csv"

if SAVE_FIGURES:
    FIG_DIR.mkdir(parents=True, exist_ok=True)

required_files = [TRAIN_TRANSACTION_PATH]
missing_required = [str(p) for p in required_files if not p.exists()]
if missing_required:
    raise FileNotFoundError(
        "Không tìm thấy file train_transaction cần thiết. "
        f"Repo root đang dùng: {REPO_ROOT}\n"
        f"Thiếu: {missing_required}\n"
        "Hãy kiểm tra lại path hoặc upload đúng thư mục raw_data/."
    )

display(Markdown(f"**Repo root:** `{REPO_ROOT}`"))
display(Markdown(f"**Raw data dir:** `{RAW_DATA_DIR}`"))

In [ ]:
def format_bytes(num_bytes):
    if num_bytes is None or pd.isna(num_bytes):
        return np.nan
    value = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if value < 1024 or unit == "TB":
            return f"{value:,.2f} {unit}"
        value /= 1024

def file_info(path):
    exists = path.exists()
    return {
        "file": path.name,
        "exists": exists,
        "size_mb": round(path.stat().st_size / (1024 ** 2), 2) if exists else np.nan,
        "path": str(path.relative_to(REPO_ROOT)) if exists else str(path),
    }

def maybe_savefig(name):
    if SAVE_FIGURES:
        plt.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=160)

def read_csv_header(path):
    if not path.exists():
        return None
    return pd.read_csv(path, nrows=0, low_memory=False)

def read_csv_full(path, nrows=None):
    if not path.exists():
        return None
    return pd.read_csv(path, nrows=nrows, low_memory=False)

def reduce_memory_usage(df, category_ratio_threshold=0.30, max_categories=200):
    start_mem = df.memory_usage(deep=True).sum()

    for col in df.columns:
        col_type = df[col].dtype

        if pd.api.types.is_datetime64_any_dtype(col_type):
            continue

        if pd.api.types.is_integer_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        elif pd.api.types.is_float_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="float")
        elif pd.api.types.is_object_dtype(col_type):
            nunique = df[col].nunique(dropna=False)
            ratio = nunique / max(len(df), 1)
            if nunique <= max_categories or ratio <= category_ratio_threshold:
                df[col] = df[col].astype("category")

    end_mem = df.memory_usage(deep=True).sum()
    reduction = (start_mem - end_mem) / start_mem if start_mem > 0 else 0
    return df, start_mem, end_mem, reduction

def detect_target_column(df, manual_target=None):
    if manual_target and manual_target in df.columns:
        return manual_target

    candidates = ["isFraud", "fraud", "Fraud", "class", "Class", "target", "Target", "label", "Label"]
    for c in candidates:
        if c in df.columns:
            return c

    binary_like = []
    for c in df.columns:
        uniq = pd.Series(df[c]).dropna().unique()
        if 1 < len(uniq) <= 2:
            binary_like.append(c)

    if len(binary_like) == 1:
        return binary_like[0]

    raise ValueError(
        "Không detect được target column tự động. "
        "Hãy set MANUAL_TARGET_COL ở cell config."
    )

def detect_id_column(df, manual_id=None):
    if manual_id and manual_id in df.columns:
        return manual_id

    preferred = ["TransactionID", "transaction_id", "TransactionId", "id", "ID"]
    for c in preferred:
        if c in df.columns:
            return c

    for c in df.columns:
        if str(c).lower().endswith("id"):
            return c

    return None

def infer_positive_class(y):
    counts = y.value_counts(dropna=False)
    if set(counts.index.tolist()) == {0, 1}:
        return 1
    return counts.idxmin()

def feature_prefix(col_name):
    match = re.match(r"([A-Za-z_]+)", str(col_name))
    return match.group(1) if match else str(col_name)

def get_feature_type_lists(df, target_col, id_col=None):
    excluded = {target_col}
    if id_col:
        excluded.add(id_col)

    numeric_cols = [c for c in df.select_dtypes(include=np.number).columns if c not in excluded]
    categorical_cols = [c for c in df.columns if c not in excluded and c not in numeric_cols]
    return numeric_cols, categorical_cols

def safe_sample(df, n, random_state=RANDOM_STATE):
    if len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=random_state)

def balanced_class_sample(df, target_col, per_class=10_000, random_state=RANDOM_STATE):
    counts = df[target_col].value_counts(dropna=False)
    parts = []
    for label in counts.index:
        part = df[df[target_col] == label]
        if len(part) > per_class:
            part = part.sample(per_class, random_state=random_state)
        parts.append(part)
    return pd.concat(parts, axis=0).sample(frac=1, random_state=random_state).reset_index(drop=True)

def summarize_missing(df):
    summary = (
        df.isna().sum().rename("missing_count").to_frame()
        .assign(
            missing_pct=lambda x: x["missing_count"] / len(df),
            dtype=lambda x: [str(df[c].dtype) for c in x.index],
            nunique=lambda x: [df[c].nunique(dropna=True) for c in x.index],
        )
        .sort_values(["missing_pct", "missing_count"], ascending=[False, False])
    )
    return summary

def summarize_categorical(df, categorical_cols):
    rows = []
    for c in categorical_cols:
        s = df[c]
        mode = s.mode(dropna=True)
        top_val = mode.iloc[0] if not mode.empty else np.nan
        top_freq_pct = s.eq(top_val).mean() if pd.notna(top_val) else np.nan

        rows.append({
            "feature": c,
            "dtype": str(s.dtype),
            "nunique": s.nunique(dropna=True),
            "missing_pct": s.isna().mean(),
            "top_value": top_val,
            "top_freq_pct": top_freq_pct,
        })

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(["missing_pct", "nunique"], ascending=[False, False])

def numeric_anomaly_table(df, numeric_cols):
    rows = []
    for c in numeric_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        non_null = s.notna().sum()
        if non_null == 0:
            continue

        valid = s.dropna()
        rows.append({
            "feature": c,
            "missing_pct": s.isna().mean(),
            "zero_pct": (s == 0).sum() / len(s),
            "negative_pct": (s < 0).sum() / len(s),
            "skew": valid.skew() if valid.nunique() > 2 else np.nan,
            "nunique": valid.nunique(),
        })

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(["negative_pct", "zero_pct", "missing_pct"], ascending=False)

def choose_priority_features(df, numeric_cols, categorical_cols):
    numeric_priority = [
        "TransactionAmt", "TransactionDT", "card1", "card2", "card3", "card5",
        "addr1", "addr2", "dist1", "dist2", "C1", "C2", "C3", "C5", "C13",
        "D1", "D2", "D4", "D10", "D15"
    ]
    categorical_priority = [
        "ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain",
        "DeviceType", "DeviceInfo", "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9"
    ]

    selected_numeric = [c for c in numeric_priority if c in numeric_cols]
    selected_categorical = [c for c in categorical_priority if c in categorical_cols]
    return selected_numeric, selected_categorical

def rank_numeric_features_by_separation(df, numeric_cols, target_col, positive_class=None, sample_per_class=10_000, top_n=20):
    if positive_class is None:
        positive_class = infer_positive_class(df[target_col])

    min_non_null = max(100, int(0.005 * len(df)))
    usable_cols = [
        c for c in numeric_cols
        if df[c].notna().sum() >= min_non_null and df[c].nunique(dropna=True) > 1
    ]

    if not usable_cols:
        return pd.DataFrame()

    working = balanced_class_sample(df[[target_col] + usable_cols].copy(), target_col, per_class=sample_per_class)

    rows = []
    for c in usable_cols:
        s = pd.to_numeric(working[c], errors="coerce")
        pos = s[working[target_col] == positive_class].dropna()
        neg = s[working[target_col] != positive_class].dropna()

        if len(pos) < 50 or len(neg) < 50:
            continue

        if ks_2samp is not None:
            try:
                ks_stat = ks_2samp(pos, neg).statistic
            except Exception:
                ks_stat = np.nan
        else:
            ks_stat = np.nan

        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        scale = iqr if pd.notna(iqr) and iqr != 0 else s.std()
        norm_median_gap = abs(pos.median() - neg.median()) / scale if pd.notna(scale) and scale != 0 else np.nan

        rows.append({
            "feature": c,
            "ks_stat": ks_stat,
            "norm_median_gap": norm_median_gap,
            "missing_pct": df[c].isna().mean(),
            "fraud_mean": pos.mean(),
            "nonfraud_mean": neg.mean(),
            "fraud_median": pos.median(),
            "nonfraud_median": neg.median(),
        })

    sep = pd.DataFrame(rows)
    if sep.empty:
        return sep

    sep = sep.sort_values(["ks_stat", "norm_median_gap"], ascending=False).reset_index(drop=True)
    return sep.head(top_n)

def pick_numeric_features_for_univariate(df, numeric_cols, target_col, max_features=8):
    priority_numeric, _ = choose_priority_features(df, numeric_cols, [])
    sep = rank_numeric_features_by_separation(df, numeric_cols, target_col, top_n=12)
    skew_table = numeric_anomaly_table(df, numeric_cols)

    skewed = []
    if not skew_table.empty and "skew" in skew_table.columns:
        skewed = (
            skew_table.assign(abs_skew=lambda x: x["skew"].abs())
            .sort_values("abs_skew", ascending=False)["feature"]
            .head(10)
            .tolist()
        )

    chosen = []
    for lst in [priority_numeric, sep["feature"].tolist() if not sep.empty else [], skewed]:
        for c in lst:
            if c in numeric_cols and c not in chosen:
                chosen.append(c)

    return chosen[:max_features], sep, skew_table

def pick_categorical_features_for_plot(df, categorical_cols, max_features=5):
    _, priority_cat = choose_priority_features(df, [], categorical_cols)

    eligible = []
    for c in categorical_cols:
        nunique = df[c].nunique(dropna=True)
        non_null_ratio = df[c].notna().mean()
        if 2 <= nunique <= 25 and non_null_ratio >= 0.05:
            eligible.append(c)

    chosen = []
    for c in priority_cat + eligible:
        if c in categorical_cols and c not in chosen:
            chosen.append(c)

    return chosen[:max_features]

def plot_univariate_numeric(df, features, sample_size=120_000):
    if not features:
        print("Không có numerical feature phù hợp để vẽ.")
        return

    plot_df = safe_sample(df[features], sample_size)

    for c in features:
        s = pd.to_numeric(plot_df[c], errors="coerce").dropna()
        if s.empty:
            continue

        fig, axes = plt.subplots(1, 2, figsize=(16, 4))

        if s.nunique() > 30:
            low, high = s.quantile([0.01, 0.99])
            plot_s = s.clip(lower=low, upper=high)
        else:
            plot_s = s

        axes[0].hist(plot_s, bins=50, density=True, alpha=0.7)
        try:
            if plot_s.nunique() > 5:
                plot_s.astype(float).plot(kind="kde", ax=axes[0])
        except Exception:
            pass
        axes[0].set_title(f"{c} - histogram/KDE-like view (1st-99th pct clipped nếu cần)")

        axes[1].boxplot(s, vert=False, showfliers=True)
        axes[1].set_title(f"{c} - boxplot (sampled values)")

        plt.tight_layout()
        maybe_savefig(f"univariate_{c}")
        plt.show()

def plot_numeric_class_comparison(df, features, target_col, positive_class=None, sample_per_class=10_000):
    if not features:
        print("Không có feature phù hợp để so sánh giữa fraud và non-fraud.")
        return

    if positive_class is None:
        positive_class = infer_positive_class(df[target_col])

    subset = balanced_class_sample(df[[target_col] + features].copy(), target_col, per_class=sample_per_class)

    for c in features:
        s = pd.to_numeric(subset[c], errors="coerce")
        if s.notna().sum() == 0:
            continue

        plot_df = pd.DataFrame({
            c: s,
            target_col: np.where(subset[target_col] == positive_class, "Fraud", "Non-Fraud")
        }).dropna()

        fig, axes = plt.subplots(1, 2, figsize=(16, 4))

        density_df = plot_df.copy()
        if plot_df[c].nunique() > 30:
            low, high = plot_df[c].quantile([0.01, 0.99])
            density_df = plot_df[(plot_df[c] >= low) & (plot_df[c] <= high)].copy()

        neg = density_df.loc[density_df[target_col] == "Non-Fraud", c].dropna().astype(float)
        pos = density_df.loc[density_df[target_col] == "Fraud", c].dropna().astype(float)

        if len(neg) > 0:
            axes[0].hist(neg, bins=40, density=True, alpha=0.5, label="Non-Fraud")
            try:
                if pd.Series(neg).nunique() > 5:
                    pd.Series(neg).plot(kind="kde", ax=axes[0])
            except Exception:
                pass

        if len(pos) > 0:
            axes[0].hist(pos, bins=40, density=True, alpha=0.5, label="Fraud")
            try:
                if pd.Series(pos).nunique() > 5:
                    pd.Series(pos).plot(kind="kde", ax=axes[0])
            except Exception:
                pass

        axes[0].legend()
        axes[0].set_title(f"{c} - class distribution / density-like view")

        full_neg = plot_df.loc[plot_df[target_col] == "Non-Fraud", c].dropna().astype(float)
        full_pos = plot_df.loc[plot_df[target_col] == "Fraud", c].dropna().astype(float)
        axes[1].boxplot([full_neg, full_pos], labels=["Non-Fraud", "Fraud"], showfliers=True)
        axes[1].set_title(f"{c} - boxplot by class")

        plt.tight_layout()
        maybe_savefig(f"class_compare_{c}")
        plt.show()

def category_fraud_table(df, col, target_col, positive_class=None, top_n=12, min_count=200):
    if positive_class is None:
        positive_class = infer_positive_class(df[target_col])

    temp = df[[col, target_col]].copy()
    temp[col] = temp[col].astype("string").fillna("MISSING")

    vc = temp[col].value_counts()
    keep = vc[vc >= min_count].head(top_n).index.tolist()
    temp[col] = np.where(temp[col].isin(keep), temp[col], "OTHER")

    out = (
        temp.groupby(col)[target_col]
        .agg(
            transaction_count="size",
            fraud_count=lambda s: (s == positive_class).sum(),
            fraud_rate=lambda s: (s == positive_class).mean(),
        )
        .sort_values("transaction_count", ascending=False)
        .reset_index()
    )
    return out

def plot_categorical_fraud_rate(df, features, target_col, positive_class=None):
    if not features:
        print("Không có categorical feature phù hợp để trực quan hóa.")
        return

    if positive_class is None:
        positive_class = infer_positive_class(df[target_col])

    min_count = max(200, int(0.002 * len(df)))

    for c in features:
        table = category_fraud_table(
            df, c, target_col,
            positive_class=positive_class,
            top_n=CATEGORY_MAX_LEVELS_FOR_PLOT,
            min_count=min_count
        )

        if table.empty:
            continue

        fig, axes = plt.subplots(1, 2, figsize=(18, 5))

        axes[0].bar(table[c].astype(str), table["transaction_count"])
        axes[0].set_title(f"{c} - transaction count")
        axes[0].tick_params(axis="x", rotation=45)

        axes[1].bar(table[c].astype(str), table["fraud_rate"])
        axes[1].set_title(f"{c} - fraud rate")
        axes[1].tick_params(axis="x", rotation=45)

        plt.tight_layout()
        maybe_savefig(f"category_{c}")
        plt.show()

def get_correlation_pairs(corr_matrix, min_abs_corr=0.80):
    pairs = []
    cols = corr_matrix.columns.tolist()

    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            corr = corr_matrix.iloc[i, j]
            if pd.notna(corr) and abs(corr) >= min_abs_corr:
                pairs.append({
                    "feature_1": cols[i],
                    "feature_2": cols[j],
                    "corr": corr,
                })

    out = pd.DataFrame(pairs)
    if out.empty:
        return out
    return out.reindex(out["corr"].abs().sort_values(ascending=False).index).reset_index(drop=True)

def outlier_summary_iqr(df, features, target_col, positive_class=None):
    if positive_class is None:
        positive_class = infer_positive_class(df[target_col])

    rows = []
    for c in features:
        s = pd.to_numeric(df[c], errors="coerce")
        valid = s.dropna()

        if valid.nunique() < 5:
            continue

        q1, q3 = valid.quantile([0.25, 0.75])
        iqr = q3 - q1
        if pd.isna(iqr) or iqr == 0:
            continue

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        mask = (s < lower) | (s > upper)

        rows.append({
            "feature": c,
            "lower_bound": lower,
            "upper_bound": upper,
            "outlier_pct_overall": mask.mean(),
            "outlier_pct_fraud": mask[df[target_col] == positive_class].mean(),
            "outlier_pct_nonfraud": mask[df[target_col] != positive_class].mean(),
        })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out["fraud_minus_nonfraud_outlier_pct"] = out["outlier_pct_fraud"] - out["outlier_pct_nonfraud"]
    return out.sort_values("fraud_minus_nonfraud_outlier_pct", ascending=False).reset_index(drop=True)

def plot_time_analysis_transactiondt(df, target_col, transactiondt_col="TransactionDT", positive_class=None):
    if positive_class is None:
        positive_class = infer_positive_class(df[target_col])

    temp = df[[transactiondt_col, target_col]].copy().dropna()
    if temp.empty:
        print("Không có dữ liệu hợp lệ để phân tích theo thời gian.")
        return None, None

    temp["pseudo_day"] = (temp[transactiondt_col] // 86400).astype(int)
    temp["pseudo_hour_of_day"] = ((temp[transactiondt_col] // 3600) % 24).astype(int)

    day_stats = (
        temp.groupby("pseudo_day")[target_col]
        .agg(
            transaction_count="size",
            fraud_count=lambda s: (s == positive_class).sum(),
            fraud_rate=lambda s: (s == positive_class).mean(),
        )
        .reset_index()
    )

    hour_stats = (
        temp.groupby("pseudo_hour_of_day")[target_col]
        .agg(
            transaction_count="size",
            fraud_rate=lambda s: (s == positive_class).mean(),
        )
        .reset_index()
    )

    fig, axes = plt.subplots(3, 1, figsize=(16, 14))

    axes[0].plot(day_stats["pseudo_day"], day_stats["transaction_count"])
    axes[0].set_title("Transaction volume by pseudo-day")

    axes[1].plot(day_stats["pseudo_day"], day_stats["fraud_count"])
    axes[1].set_title("Fraud count by pseudo-day")

    axes[2].plot(day_stats["pseudo_day"], day_stats["fraud_rate"])
    axes[2].set_title("Fraud rate by pseudo-day")
    axes[2].set_ylim(bottom=0)

    plt.tight_layout()
    maybe_savefig("time_pseudo_day")
    plt.show()

    plt.figure(figsize=(14, 4))
    plt.bar(hour_stats["pseudo_hour_of_day"].astype(str), hour_stats["fraud_rate"])
    plt.title("Fraud rate by pseudo-hour-of-day")
    plt.ylim(bottom=0)
    plt.tight_layout()
    maybe_savefig("time_pseudo_hour")
    plt.show()

    return day_stats, hour_stats

## 1. Setup + Load Dataset

Ở repo hiện tại, file phù hợp nhất cho EDA có target là **`train_transaction.csv.gz`**.  
Nếu có **`train_identity.csv`**, notebook sẽ merge thêm theo `TransactionID` để EDA có chiều sâu hơn.

`test_transaction` và `test_identity` vẫn được kiểm tra schema ở mức header để xem train/test có khớp cột hay không, nhưng **EDA chính sẽ chạy trên training data** vì chỉ training split mới có target phục vụ phân tích fraud vs non-fraud.

In [ ]:
file_inventory = pd.DataFrame([
    file_info(TRAIN_TRANSACTION_PATH),
    file_info(TRAIN_IDENTITY_PATH),
    file_info(TEST_TRANSACTION_PATH),
    file_info(TEST_IDENTITY_PATH),
    file_info(SAMPLE_SUBMISSION_PATH),
])
display(file_inventory)

train_tx_header = read_csv_header(TRAIN_TRANSACTION_PATH)
train_id_header = read_csv_header(TRAIN_IDENTITY_PATH) if TRAIN_IDENTITY_PATH.exists() else None
test_tx_header = read_csv_header(TEST_TRANSACTION_PATH) if TEST_TRANSACTION_PATH.exists() else None
test_id_header = read_csv_header(TEST_IDENTITY_PATH) if TEST_IDENTITY_PATH.exists() else None

schema_overview = pd.DataFrame([
    {"table": "train_transaction", "n_columns": len(train_tx_header.columns) if train_tx_header is not None else np.nan},
    {"table": "train_identity", "n_columns": len(train_id_header.columns) if train_id_header is not None else np.nan},
    {"table": "test_transaction", "n_columns": len(test_tx_header.columns) if test_tx_header is not None else np.nan},
    {"table": "test_identity", "n_columns": len(test_id_header.columns) if test_id_header is not None else np.nan},
])
display(schema_overview)

if train_tx_header is not None and test_tx_header is not None:
    only_in_train_tx = sorted(set(train_tx_header.columns) - set(test_tx_header.columns))
    only_in_test_tx = sorted(set(test_tx_header.columns) - set(train_tx_header.columns))

    display(Markdown(
        "**Transaction schema diff**  \n"
        f"- Only in `train_transaction`: `{only_in_train_tx[:20]}`  \n"
        f"- Only in `test_transaction`: `{only_in_test_tx[:20]}`"
    ))

if train_id_header is not None and test_id_header is not None:
    only_in_train_id = sorted(set(train_id_header.columns) - set(test_id_header.columns))
    only_in_test_id = sorted(set(test_id_header.columns) - set(train_id_header.columns))

    display(Markdown(
        "**Identity schema diff**  \n"
        f"- Only in `train_identity`: `{only_in_train_id[:20]}`  \n"
        f"- Only in `test_identity`: `{only_in_test_id[:20]}`"
    ))

In [ ]:
# Load training transaction table
train_transaction = read_csv_full(TRAIN_TRANSACTION_PATH, nrows=NROWS)
train_transaction, tx_mem_before, tx_mem_after, tx_reduction = reduce_memory_usage(train_transaction)

# Optionally load identity table
train_identity = None
id_mem_before = id_mem_after = id_reduction = np.nan
if LOAD_IDENTITY and TRAIN_IDENTITY_PATH.exists():
    train_identity = read_csv_full(TRAIN_IDENTITY_PATH, nrows=NROWS)
    train_identity, id_mem_before, id_mem_after, id_reduction = reduce_memory_usage(train_identity)

TARGET_COL = detect_target_column(train_transaction, manual_target=MANUAL_TARGET_COL)
ID_COL = detect_id_column(train_transaction, manual_id=MANUAL_ID_COL)

identity_match_rate = None
if train_identity is not None and ID_COL and ID_COL in train_identity.columns:
    identity_match_rate = train_transaction[ID_COL].isin(train_identity[ID_COL]).mean()

if MERGE_IDENTITY and train_identity is not None and ID_COL and ID_COL in train_identity.columns:
    try:
        df = train_transaction.merge(train_identity, on=ID_COL, how="left", validate="one_to_one")
    except Exception as e:
        print(f"Merge validate='one_to_one' failed ({e}). Fallback sang standard left join.")
        df = train_transaction.merge(train_identity, on=ID_COL, how="left")
    data_mode = f"train_transaction LEFT JOIN train_identity on {ID_COL}"
else:
    df = train_transaction.copy()
    data_mode = "train_transaction only"

if TARGET_COL in df.columns:
    uniq_target = set(pd.Series(df[TARGET_COL]).dropna().unique().tolist())
    if uniq_target.issubset({0, 1}):
        df[TARGET_COL] = df[TARGET_COL].astype("int8")

POSITIVE_CLASS = infer_positive_class(df[TARGET_COL])

load_summary = pd.DataFrame([
    {
        "table": "train_transaction",
        "shape": train_transaction.shape,
        "memory_before": format_bytes(tx_mem_before),
        "memory_after": format_bytes(tx_mem_after),
        "reduction_pct": round(tx_reduction * 100, 2),
    },
    {
        "table": "train_identity",
        "shape": train_identity.shape if train_identity is not None else None,
        "memory_before": format_bytes(id_mem_before) if pd.notna(id_mem_before) else None,
        "memory_after": format_bytes(id_mem_after) if pd.notna(id_mem_after) else None,
        "reduction_pct": round(id_reduction * 100, 2) if pd.notna(id_reduction) else None,
    },
    {
        "table": "working_df",
        "shape": df.shape,
        "memory_before": None,
        "memory_after": format_bytes(df.memory_usage(deep=True).sum()),
        "reduction_pct": None,
    },
])
display(load_summary)

display(Markdown(
    f"**Target detected:** `{TARGET_COL}`  \n"
    f"**ID column detected:** `{ID_COL}`  \n"
    f"**Positive/Fraud class used in analysis:** `{POSITIVE_CLASS}`  \n"
    f"**Working mode:** `{data_mode}`"
))

if identity_match_rate is not None:
    display(Markdown(f"**Identity coverage over transaction table:** `{identity_match_rate:.2%}`"))

numeric_cols, categorical_cols = get_feature_type_lists(df, TARGET_COL, ID_COL)
print(f"Numerical features: {len(numeric_cols):,}")
print(f"Categorical features: {len(categorical_cols):,}")

## 2. Dataset Overview

Phần này trả lời các câu hỏi nền tảng:
- Dataset có bao nhiêu dòng / cột?
- Kiểu dữ liệu thế nào?
- Có đúng target fraud hay không?
- Có những nhóm cột lớn nào?
- Có dấu hiệu dataset thuộc dạng nhiều nhóm feature như `V`, `C`, `D`, `M`, `id_` hay không?

In [ ]:
overview = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "memory_usage",
        "numeric_features",
        "categorical_features",
        "target_col",
        "id_col",
    ],
    "value": [
        f"{df.shape[0]:,}",
        f"{df.shape[1]:,}",
        format_bytes(df.memory_usage(deep=True).sum()),
        f"{len(numeric_cols):,}",
        f"{len(categorical_cols):,}",
        TARGET_COL,
        ID_COL,
    ]
})
display(overview)

display(df.head())

dtype_summary = (
    df.dtypes.astype(str)
    .value_counts()
    .rename_axis("dtype")
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
display(dtype_summary)

prefix_summary = (
    pd.Series([feature_prefix(c) for c in df.columns], name="prefix")
    .value_counts()
    .rename_axis("prefix")
    .reset_index(name="n_columns")
)
display(prefix_summary.head(25))

print("Tip: nếu thấy các prefix như V, C, D, M, card, addr, id_, đây thường là dấu hiệu dataset có nhiều nhóm feature bán-ẩn danh / engineered-like.")

## 3. Target Analysis

Đây là phần quan trọng nhất cho bài toán fraud detection.

Với fraud detection, chỉ nhìn accuracy là rất dễ bị đánh lừa vì class fraud thường rất hiếm.  
Do đó notebook này sẽ nhấn mạnh:
- fraud ratio
- mức độ mất cân bằng lớp
- baseline nếu luôn đoán non-fraud
- vì sao **Recall, Precision, F1-score, PR-AUC** phải được ưu tiên

In [ ]:
target_counts = df[TARGET_COL].value_counts(dropna=False).sort_index()
target_ratio = df[TARGET_COL].value_counts(dropna=False, normalize=True).sort_index()

fraud_count = int((df[TARGET_COL] == POSITIVE_CLASS).sum())
nonfraud_count = int((df[TARGET_COL] != POSITIVE_CLASS).sum())
fraud_ratio = fraud_count / len(df)
imbalance_ratio = nonfraud_count / max(fraud_count, 1)

target_table = pd.DataFrame({
    "class": target_counts.index.tolist(),
    "count": target_counts.values,
    "ratio": target_ratio.values,
})
display(target_table)

target_count_plot_df = target_table.copy()
target_count_plot_df["class"] = target_count_plot_df["class"].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(target_count_plot_df["class"], target_count_plot_df["count"])
axes[0].set_title("Class distribution (count)")
axes[0].set_xlabel(TARGET_COL)
axes[0].set_ylabel("Count")

axes[1].bar(target_count_plot_df["class"], target_count_plot_df["ratio"])
axes[1].set_title("Class distribution (ratio)")
axes[1].set_xlabel(TARGET_COL)
axes[1].set_ylabel("Ratio")

for ax, value_col in zip(axes, ["count", "ratio"]):
    for x, value in zip(target_count_plot_df["class"].tolist(), target_count_plot_df[value_col].tolist()):
        label = f"{value:,.4f}" if value_col == "ratio" else f"{int(value):,}"
        ax.annotate(
            label,
            (x, value),
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.tight_layout()
maybe_savefig("target_distribution")
plt.show()

display(Markdown(
    f"""
**Interpretation**

- Fraud class = `{POSITIVE_CLASS}`
- Fraud count = `{fraud_count:,}`
- Fraud ratio = `{fraud_ratio:.2%}`
- Non-fraud / Fraud imbalance ratio ~= `{imbalance_ratio:.2f}:1`
- Nếu model luôn đoán **non-fraud**, accuracy baseline sẽ khoảng `{1 - fraud_ratio:.2%}`

**Implication for modeling**
- Accuracy **không đủ tốt** để đánh giá model
- Cần theo dõi **Recall** để không bỏ sót fraud
- Cần theo dõi **Precision** để tránh bắn quá nhiều false positive
- **F1-score** hữu ích khi cần cân bằng Recall và Precision
- **PR-AUC** thường phù hợp hơn ROC-AUC trong bối cảnh class imbalance mạnh
"""
))

## 4. Basic Data Inspection trong EDA

Phần này chưa phải một Data Quality module riêng, nhưng vẫn kiểm tra kỹ các điểm có ảnh hưởng mạnh đến EDA và modeling:
- missing values
- duplicate rows
- duplicate ID
- constant / near-constant features
- infinite values
- thống kê mô tả numerical
- snapshot categorical
- các dấu hiệu giá trị bất thường ở mức EDA

In [ ]:
missing_summary = summarize_missing(df)

duplicate_rows = int(df.duplicated().sum())
duplicate_id_rows = int(df[ID_COL].duplicated().sum()) if ID_COL is not None else np.nan

constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]

quasi_constant_cols = []
for c in df.columns:
    vc = df[c].value_counts(dropna=False, normalize=True)
    if not vc.empty and vc.iloc[0] >= 0.995:
        quasi_constant_cols.append(c)

inf_count = 0
if numeric_cols:
    inf_count = int(np.isinf(df[numeric_cols].to_numpy(dtype="float64", na_value=np.nan)).sum())

inspection_summary = pd.DataFrame([
    {"check": "duplicate full rows", "value": duplicate_rows},
    {"check": f"duplicate {ID_COL}" if ID_COL else "duplicate ID", "value": duplicate_id_rows},
    {"check": "constant columns", "value": len(constant_cols)},
    {"check": "quasi-constant columns (top value >= 99.5%)", "value": len(quasi_constant_cols)},
    {"check": "columns with >= 50% missing", "value": int((missing_summary["missing_pct"] >= 0.50).sum())},
    {"check": f"columns with >= {MISSINGNESS_ALERT_THRESHOLD:.0%} missing", "value": int((missing_summary["missing_pct"] >= MISSINGNESS_ALERT_THRESHOLD).sum())},
    {"check": "infinite values across numeric columns", "value": inf_count},
])
display(inspection_summary)

display(Markdown("**Top columns with highest missingness**"))
display(missing_summary.head(20))

if constant_cols:
    print("Constant columns (first 20):", constant_cols[:20])

if quasi_constant_cols:
    print("Quasi-constant columns (first 20):", quasi_constant_cols[:20])

In [ ]:
numeric_summary = pd.DataFrame()
if numeric_cols:
    numeric_summary = df[numeric_cols].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    ).T
    numeric_summary["missing_pct"] = df[numeric_cols].isna().mean()
    numeric_summary["zero_pct"] = (df[numeric_cols] == 0).sum() / len(df)
    display(Markdown("**Numerical summary (sample view)**"))
    display(numeric_summary.sort_values("missing_pct", ascending=False).head(20))

categorical_summary = summarize_categorical(df, categorical_cols)
if not categorical_summary.empty:
    display(Markdown("**Categorical summary (sample view)**"))
    display(categorical_summary.head(20))

anomaly_table = numeric_anomaly_table(df, numeric_cols)
if not anomaly_table.empty:
    display(Markdown("**Numeric anomaly snapshot (missing / zero / negative / skew)**"))
    display(anomaly_table.head(20))

print("Full tables are stored in variables: numeric_summary, categorical_summary, anomaly_table")

## 5. Univariate Analysis

Vì fraud detection datasets thường có rất nhiều feature, notebook không vẽ toàn bộ tất cả biến vì sẽ rất nặng.  
Thay vào đó, notebook chọn feature theo hướng thực dụng:
1. ưu tiên các biến thường có ý nghĩa domain (`TransactionAmt`, `TransactionDT`, `card*`, `addr*`, `C*`, `D*`, ...)
2. bổ sung các feature có dấu hiệu tách fraud vs non-fraud tốt
3. bổ sung các feature có skew/outlier mạnh để tránh bỏ sót tín hiệu quan trọng

In [ ]:
selected_numeric_features, feature_separation_table, skew_table = pick_numeric_features_for_univariate(
    df, numeric_cols, TARGET_COL, max_features=MAX_NUMERIC_PLOTS
)
selected_categorical_features = pick_categorical_features_for_plot(
    df, categorical_cols, max_features=MAX_CATEGORICAL_FEATURES
)

display(Markdown(f"**Selected numerical features for deeper EDA:** `{selected_numeric_features}`"))
display(Markdown(f"**Selected categorical features for class comparison plots:** `{selected_categorical_features}`"))

if not feature_separation_table.empty:
    display(Markdown("**Top numerical features by fraud vs non-fraud separation**"))
    display(feature_separation_table.head(15))

if not skew_table.empty:
    display(Markdown("**Top skewed numerical features (absolute skew)**"))
    display(
        skew_table.assign(abs_skew=lambda x: x["skew"].abs())
        .sort_values("abs_skew", ascending=False)
        .head(15)
    )

In [ ]:
plot_univariate_numeric(df, selected_numeric_features, sample_size=PLOT_SAMPLE_SIZE)

if not anomaly_table.empty and selected_numeric_features:
    uni_summary = anomaly_table.set_index("feature").loc[
        [c for c in selected_numeric_features if c in anomaly_table["feature"].values],
        ["missing_pct", "zero_pct", "negative_pct", "skew", "nunique"]
    ]
    display(Markdown("**Quick summary for plotted numerical features**"))
    display(uni_summary)

### Univariate takeaway

Khi xem univariate plots cho fraud detection, cần chú ý đặc biệt:
- feature có phân phối rất lệch (skewed)
- feature có nhiều giá trị 0
- feature có đuôi dài / outliers mạnh
- feature bị missing nhiều
- feature có boxplot rất rộng hoặc nhiều extreme values

Những đặc điểm này không có nghĩa là feature xấu.  
Trong fraud detection, chính các giá trị “kỳ lạ” đôi khi lại là tín hiệu mạnh để bắt fraud.

## 6. Fraud vs Non-Fraud Comparison

Đây là phần cốt lõi cho bài toán fraud detection.

Mục tiêu:
- xem feature nào có phân phối khác nhau rõ giữa fraud và non-fraud
- xem missingness có thể là tín hiệu không
- xem categorical level nào có fraud rate cao bất thường
- rút ra feature nào có thể hữu ích cho Recall / Precision / PR-AUC sau này

Để biểu đồ dễ đọc hơn, notebook sẽ dùng **balanced sample cho phần trực quan hóa class comparison** nếu dữ liệu quá lớn.

In [ ]:
top_compare_features = (
    feature_separation_table["feature"].head(MAX_COMPARE_PLOTS).tolist()
    if not feature_separation_table.empty
    else selected_numeric_features[:MAX_COMPARE_PLOTS]
)

display(Markdown(f"**Top numerical features chosen for fraud vs non-fraud plots:** `{top_compare_features}`"))

missing_signal_rows = []
for c in top_compare_features:
    missing_fraud = df.loc[df[TARGET_COL] == POSITIVE_CLASS, c].isna().mean()
    missing_nonfraud = df.loc[df[TARGET_COL] != POSITIVE_CLASS, c].isna().mean()
    missing_signal_rows.append({
        "feature": c,
        "missing_pct_fraud": missing_fraud,
        "missing_pct_nonfraud": missing_nonfraud,
        "fraud_minus_nonfraud_missing_pct": missing_fraud - missing_nonfraud,
    })

missing_signal_table = pd.DataFrame(missing_signal_rows)
if not missing_signal_table.empty:
    display(Markdown("**Missingness signal by class for top comparison features**"))
    display(
        missing_signal_table.sort_values(
            "fraud_minus_nonfraud_missing_pct",
            key=lambda s: s.abs(),
            ascending=False
        )
    )

plot_numeric_class_comparison(
    df,
    top_compare_features,
    TARGET_COL,
    positive_class=POSITIVE_CLASS,
    sample_per_class=BALANCED_CLASS_SAMPLE_PER_CLASS
)

In [ ]:
if selected_categorical_features:
    display(Markdown("**Categorical fraud-rate tables**"))
    for c in selected_categorical_features:
        table = category_fraud_table(
            df, c, TARGET_COL,
            positive_class=POSITIVE_CLASS,
            top_n=CATEGORY_MAX_LEVELS_FOR_PLOT,
            min_count=max(200, int(0.002 * len(df)))
        )
        display(Markdown(f"#### {c}"))
        display(table)

    plot_categorical_fraud_rate(
        df,
        selected_categorical_features,
        TARGET_COL,
        positive_class=POSITIVE_CLASS
    )
else:
    print("Không có categorical feature phù hợp để so sánh theo class.")

### Fraud vs Non-Fraud takeaway

Ở bước này, hãy chú ý các kiểu tín hiệu sau:
- median / density của fraud lệch đáng kể so với non-fraud
- boxplot của fraud có dải giá trị khác rõ rệt
- missing rate giữa hai class khác nhau
- một số category có fraud rate cao hơn nền chung rất nhiều

Các feature như vậy thường đáng chú ý cho bước modeling tiếp theo, đặc biệt khi muốn cải thiện:
- **Recall** mà không làm Precision sụp quá mạnh
- **PR-AUC** trong bài toán class imbalance

## 7. Correlation Analysis

Với fraud detection dataset nhiều cột, heatmap toàn bộ ma trận tương quan thường rất khó đọc.  
Notebook này dùng cách thực dụng hơn:
- lấy **subset numerical features đáng chú ý nhất**
- vẽ heatmap trên subset đó
- liệt kê:
  - feature tương quan mạnh với target
  - các cặp feature tương quan mạnh với nhau

Điều này hữu ích để:
- phát hiện redundancy
- tránh hiểu nhầm nhiều feature “khác nhau” nhưng thực ra gần như cùng tín hiệu
- chuẩn bị cho bước feature selection / regularization / model choice

In [ ]:
corr_feature_candidates = []
for c in top_compare_features + selected_numeric_features:
    if c not in corr_feature_candidates:
        corr_feature_candidates.append(c)

if not feature_separation_table.empty:
    for c in feature_separation_table["feature"].tolist():
        if c not in corr_feature_candidates:
            corr_feature_candidates.append(c)

corr_feature_candidates = corr_feature_candidates[:MAX_CORR_FEATURES]

corr_cols = [TARGET_COL] + [c for c in corr_feature_candidates if c in df.columns]
corr_sample = safe_sample(df[corr_cols], n=min(100_000, len(df)))

corr_matrix = corr_sample.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(max(10, len(corr_matrix) * 0.45), max(8, len(corr_matrix) * 0.45)))
im = ax.imshow(corr_matrix.values, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=90)
ax.set_yticks(range(len(corr_matrix.index)))
ax.set_yticklabels(corr_matrix.index)
ax.set_title("Correlation heatmap for selected numerical features")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
maybe_savefig("correlation_heatmap")
plt.show()

target_corr_table = pd.DataFrame()
if TARGET_COL in corr_matrix.columns:
    target_corr_table = (
        corr_matrix[[TARGET_COL]]
        .drop(index=[TARGET_COL], errors="ignore")
        .rename(columns={TARGET_COL: "corr_with_target"})
        .assign(abs_corr=lambda x: x["corr_with_target"].abs())
        .sort_values("abs_corr", ascending=False)
    )
    display(Markdown("**Top absolute correlations with target**"))
    display(target_corr_table.head(20))

pair_matrix = corr_matrix.drop(index=[TARGET_COL], columns=[TARGET_COL], errors="ignore")
high_corr_pairs = get_correlation_pairs(pair_matrix, min_abs_corr=0.80)
if not high_corr_pairs.empty:
    display(Markdown("**Highly correlated feature pairs (|corr| >= 0.80)**"))
    display(high_corr_pairs.head(20))
else:
    print("Không thấy cặp feature nào có |corr| >= 0.80 trong subset được chọn.")

## 8. Outlier Analysis

Trong fraud detection, outlier **không phải lúc nào cũng nên loại bỏ**.

Rất nhiều giao dịch fraud có thể xuất hiện như:
- amount cực lớn / cực nhỏ
- pattern khác thường
- distance / delay / count bất thường
- score hoặc aggregate feature ở vùng rìa phân phối

Vì vậy:
- notebook sẽ **phân tích outlier**
- nhưng **không tự động kết luận nên remove**
- thay vào đó sẽ xem outlier rate giữa fraud và non-fraud có khác nhau hay không

In [ ]:
outlier_features = top_compare_features[:10]
outlier_table = outlier_summary_iqr(
    df,
    outlier_features,
    TARGET_COL,
    positive_class=POSITIVE_CLASS
)

if not outlier_table.empty:
    display(Markdown("**IQR-based outlier summary**"))
    display(outlier_table)
else:
    print("Không tính được bảng outlier cho các feature đã chọn.")

In [ ]:
if not outlier_table.empty:
    top_outlier_plot_features = outlier_table["feature"].head(4).tolist()
    subset = balanced_class_sample(
        df[[TARGET_COL] + top_outlier_plot_features].copy(),
        TARGET_COL,
        per_class=min(5_000, BALANCED_CLASS_SAMPLE_PER_CLASS)
    )

    for c in top_outlier_plot_features:
        plot_df = subset[[TARGET_COL, c]].copy()
        plot_df[TARGET_COL] = np.where(plot_df[TARGET_COL] == POSITIVE_CLASS, "Fraud", "Non-Fraud")
        plot_df[c] = pd.to_numeric(plot_df[c], errors="coerce")
        plot_df = plot_df.dropna()

        if plot_df.empty:
            continue

        neg = plot_df.loc[plot_df[TARGET_COL] == "Non-Fraud", c].astype(float)
        pos = plot_df.loc[plot_df[TARGET_COL] == "Fraud", c].astype(float)

        plt.figure(figsize=(10, 4))
        plt.boxplot([neg, pos], labels=["Non-Fraud", "Fraud"], showfliers=True)
        plt.title(f"{c} - outlier behavior by class")
        plt.tight_layout()
        maybe_savefig(f"outlier_boxplot_{c}")
        plt.show()
else:
    print("Skip outlier plots vì chưa có outlier_table.")

### Outlier takeaway

Nếu feature có:
- outlier rate ở fraud cao hơn non-fraud
- boxplot fraud kéo dài hơn rõ rệt
- rất nhiều extreme values tập trung ở fraud

thì đó là tín hiệu tốt cho modeling.  
Trong case này, nên cân nhắc:
- log transform
- robust scaling
- tree-based models
- threshold tuning

thay vì remove outlier một cách cơ học.

## 9. Time-based Analysis

Nếu dataset có biến thời gian, đây là phần rất đáng xem trong fraud detection.

Notebook sẽ:
- tự tìm cột thời gian
- nếu có `TransactionDT`, sẽ phân tích:
  - transaction volume theo pseudo-time
  - fraud count theo pseudo-time
  - fraud rate theo pseudo-time
  - fraud rate theo pseudo-hour-of-day

> Với dataset kiểu IEEE-CIS, `TransactionDT` thường là **relative timedelta**, không phải timestamp thực.

In [ ]:
time_candidates = [c for c in df.columns if re.search(r"(time|date|dt|timestamp)", str(c), flags=re.I)]
display(Markdown(f"**Time-like candidates found:** `{time_candidates}`"))

day_stats = None
hour_stats = None

if "TransactionDT" in df.columns:
    day_stats, hour_stats = plot_time_analysis_transactiondt(
        df,
        TARGET_COL,
        transactiondt_col="TransactionDT",
        positive_class=POSITIVE_CLASS
    )

    if day_stats is not None and not day_stats.empty:
        display(Markdown("**Time summary by pseudo-day (head)**"))
        display(day_stats.head())

    if hour_stats is not None and not hour_stats.empty:
        display(Markdown("**Time summary by pseudo-hour-of-day**"))
        display(hour_stats)
else:
    print("Không có cột `TransactionDT`. Skip phân tích time-based đặc thù.")

## 10. EDA Summary

Cell dưới đây sẽ tổng hợp các insight quan trọng nhất theo hướng **phục vụ modeling** chứ không chỉ mô tả dữ liệu.

In [ ]:
high_missing_cols_count = int((missing_summary["missing_pct"] >= MISSINGNESS_ALERT_THRESHOLD).sum())

top_sep_features = (
    feature_separation_table["feature"].head(5).tolist()
    if not feature_separation_table.empty
    else []
)

top_target_corr_features = (
    target_corr_table.head(5).index.tolist()
    if not target_corr_table.empty
    else []
)

top_outlier_features = (
    outlier_table["feature"].head(5).tolist()
    if not outlier_table.empty
    else []
)

time_summary_line = "- Không có cột thời gian phù hợp để phân tích."
if day_stats is not None and not day_stats.empty:
    peak_day = int(day_stats.loc[day_stats["fraud_rate"].idxmax(), "pseudo_day"])
    peak_rate = float(day_stats["fraud_rate"].max())
    time_summary_line = f"- Fraud rate cao nhất trong pseudo-time xuất hiện ở **pseudo_day = {peak_day}** với fraud rate khoảng **{peak_rate:.2%}**."

summary_md = f"""
## Key EDA Insights

- Working dataframe: **{df.shape[0]:,} rows x {df.shape[1]:,} columns**
- Data mode: **{data_mode}**
- Target column: **{TARGET_COL}**
- Fraud ratio: **{fraud_ratio:.2%}**
- Class imbalance: khoảng **{imbalance_ratio:.2f}:1** (non-fraud : fraud)
- Numerical features: **{len(numeric_cols):,}**
- Categorical features: **{len(categorical_cols):,}**
- Columns có missing >= {MISSINGNESS_ALERT_THRESHOLD:.0%}: **{high_missing_cols_count:,}**
- Duplicate full rows: **{duplicate_rows:,}**
- Duplicate {ID_COL if ID_COL else "ID"}: **{duplicate_id_rows:,}**

### Most important patterns for modeling
- Top numerical features có dấu hiệu tách fraud vs non-fraud tốt: **{", ".join(top_sep_features) if top_sep_features else "chưa xác định rõ từ sample hiện tại"}**
- Top features có tương quan mạnh nhất với target trong correlation subset: **{", ".join(top_target_corr_features) if top_target_corr_features else "chưa có bảng target correlation"}**
- Top features có khác biệt outlier rate đáng chú ý giữa fraud và non-fraud: **{", ".join(top_outlier_features) if top_outlier_features else "chưa có bảng outlier"}**
{time_summary_line}

### Implications for next modeling step
- Nên dùng **stratified split / stratified CV**
- Khi đánh giá model, ưu tiên báo cáo:
  - **Recall**
  - **Precision**
  - **F1-score**
  - **PR-AUC**
- Nên có **threshold tuning**, không chỉ dựa vào default threshold = 0.5
- Không nên loại outliers một cách máy móc
- Missingness có thể là tín hiệu, không chỉ là “rác”
- Nếu nhiều feature tương quan mạnh, cần cân nhắc redundancy ở bước feature selection / regularization
"""
display(Markdown(summary_md))

# 11. DVC Assessment for This Repository

Phần này không nói DVC theo kiểu lý thuyết chung.  
Nó bám sát repo hiện tại và trả lời thẳng câu hỏi:

> **Với repo `haminh109/Fraud-Detection` ở trạng thái hiện tại, có nên dùng DVC ngay không?**

In [ ]:
repo_state = pd.DataFrame([
    {"artifact": "raw_data/", "exists": (REPO_ROOT / "raw_data").exists(), "size_mb": round(sum(p.stat().st_size for p in (REPO_ROOT / "raw_data").glob("*") if p.is_file()) / (1024 ** 2), 2) if (REPO_ROOT / "raw_data").exists() else np.nan},
    {"artifact": ".dvc/", "exists": (REPO_ROOT / ".dvc").exists(), "size_mb": np.nan},
    {"artifact": "dvc.yaml", "exists": (REPO_ROOT / "dvc.yaml").exists(), "size_mb": round((REPO_ROOT / "dvc.yaml").stat().st_size / (1024 ** 2), 4) if (REPO_ROOT / "dvc.yaml").exists() else np.nan},
    {"artifact": "params.yaml", "exists": (REPO_ROOT / "params.yaml").exists(), "size_mb": round((REPO_ROOT / "params.yaml").stat().st_size / (1024 ** 2), 4) if (REPO_ROOT / "params.yaml").exists() else np.nan},
    {"artifact": "notebooks/", "exists": (REPO_ROOT / "notebooks").exists(), "size_mb": np.nan},
    {"artifact": "src/", "exists": (REPO_ROOT / "src").exists(), "size_mb": np.nan},
    {"artifact": "models/", "exists": (REPO_ROOT / "models").exists(), "size_mb": np.nan},
    {"artifact": "data/interim/", "exists": (REPO_ROOT / "data" / "interim").exists(), "size_mb": np.nan},
    {"artifact": "data/processed/", "exists": (REPO_ROOT / "data" / "processed").exists(), "size_mb": np.nan},
])
display(repo_state)

display(Markdown(
    "Notebook conclusion below sẽ chọn rõ **một trong ba hướng**:\n"
    "1. Nên dùng DVC ngay\n"
    "2. Nên làm EDA trước, DVC sau\n"
    "3. Chưa cần DVC ở giai đoạn này"
))

## DVC Decision: **2. Nên làm EDA trước, DVC sau**

### Vì sao đây là quyết định phù hợp nhất với repo hiện tại?

**Repo hiện tại còn ở giai đoạn rất sớm**
- Ở snapshot hiện tại, repo chủ yếu là `raw_data/` và `README.md`
- Chưa thấy cấu trúc pipeline rõ ràng cho:
  - `data/interim`
  - `data/processed`
  - `models`
  - `params.yaml`
  - `dvc.yaml`
  - `src/`

**Raw dataset đã được đưa trực tiếp vào Git**
- Điều này nghĩa là nếu ép migrate sang DVC ngay lập tức, bạn sẽ phải xử lý thêm phần chuyển đổi cấu trúc repo
- Với project sinh viên, làm việc đó **ngay trước khi EDA ổn định** thường chưa tạo ra nhiều giá trị tức thì

**Nhưng DVC sẽ rất hữu ích ngay sau bước EDA**
- Ngay khi bạn bắt đầu tạo ra:
  - merged/interim dataset
  - processed features
  - saved models
  - validation predictions
  - metrics / experiment artifacts
- Nếu các artifact này tiếp tục được commit bằng Git thường, repo sẽ nhanh chóng lộn xộn và nặng

### Kết luận thực dụng
- **Làm xong EDA này trước**
- Sau đó **tích hợp DVC ngay trước khi bước sang feature engineering / training / model artifact**
- Nghĩa là: **không bỏ qua DVC**, nhưng cũng **không cần cưỡng ép migrate mọi thứ ngay phút này**

## DVC giúp gì cho project fraud detection này?

Nếu bạn tiếp tục theo hướng MLOps tối thiểu, DVC sẽ giúp:
1. Version hóa các dataset sinh ra sau EDA mà không nhồi file lớn vào Git
2. Đồng bộ model artifact giữa các thành viên nhóm
3. Tách rõ:
   - code / notebook / config -> Git
   - data / model / artifact lớn -> DVC
4. Giúp tái lập kết quả tốt hơn khi:
   - đổi split
   - đổi preprocessing
   - đổi feature set
   - đổi model
5. Giảm rủi ro “mỗi máy một bản dữ liệu hơi khác nhau”

## Nên track gì bằng DVC, và gì bằng Git thường?

### Track bằng **DVC**
- `data/interim/`
- `data/processed/`
- `models/`
- prediction files lớn
- training outputs nặng
- về sau có thể là `data/raw/` nếu bạn quyết định migrate raw dataset khỏi Git

### Track bằng **Git thường**
- `README.md`
- notebook EDA
- code trong `src/`
- `requirements.txt`
- `params.yaml`
- `dvc.yaml`
- các file `.dvc`
- hình ảnh nhẹ / báo cáo nhẹ / markdown
- config không chứa secret

### Không nên commit vào Git
- `.dvc/cache/`
- `.dvc/config.local`
- secret / credential
- model binary lớn
- processed data lớn

## Mức triển khai DVC tối thiểu, hợp lý cho project sinh viên

Đây là mức **đủ dùng**, không quá over-engineer:

```text
Fraud-Detection/
├─ raw_data/                          # giữ tạm thời như hiện tại ở giai đoạn sau EDA
├─ notebooks/
│  └─ 01_eda_fraud_detection.ipynb
├─ data/
│  ├─ interim/                        # DVC
│  └─ processed/                      # DVC
├─ models/                            # DVC
├─ reports/
│  ├─ figures/
│  └─ metrics/
├─ src/
│  ├─ data/
│  ├─ features/
│  ├─ modeling/
│  └─ utils/
├─ .dvc/
├─ dvc.yaml                           # thêm sau khi bắt đầu pipeline
├─ params.yaml                        # thêm khi bắt đầu config hóa pipeline
├─ requirements.txt
└─ README.md
```

### Cách hiểu thực dụng
- **Ngắn hạn**: cứ giữ `raw_data/` như hiện tại để tránh phải “phẫu thuật repo” ngay
- **Trung hạn**: bắt đầu DVC từ `data/interim/`, `data/processed/`, `models/`
- **Khi project ổn hơn**: có thể chuyển `raw_data/` sang `data/raw/` rồi track bằng DVC

## Cách triển khai DVC thực tế cho repo này

### Bước 1 — Cài DVC
Nếu chỉ dùng local/shared-folder remote:

```bash
pip install dvc
```

Nếu muốn dùng Google Drive remote:

```bash
pip install "dvc[gdrive]"
```

### Bước 2 — Init DVC ở repo root

```bash
dvc init
git add .dvc .gitignore
git commit -m "Initialize DVC"
```

### Bước 3 — Tạo cấu trúc tối thiểu cho giai đoạn sau EDA

```bash
mkdir -p notebooks data/interim data/processed models reports/figures reports/metrics src
```

### Bước 4 — Bắt đầu track artifact sinh ra sau EDA
Ví dụ sau khi bạn lưu merged dataset hoặc processed dataset:

```bash
dvc add data/interim/train_merged.parquet
dvc add data/processed/train_features.parquet
dvc add models/lgbm_baseline.joblib
```

Sau đó commit metadata vào Git:

```bash
git add .gitignore data/interim/train_merged.parquet.dvc data/processed/train_features.parquet.dvc models/lgbm_baseline.joblib.dvc
git commit -m "Track merged data, processed features, and baseline model with DVC"
```

## Chọn remote storage nào là hợp lý cho nhóm sinh viên?

### Lựa chọn 1 — **Local / shared folder remote** (đơn giản nhất)
Phù hợp nếu:
- nhóm làm việc chủ yếu trên 1 máy chính / 1 ổ dùng chung / 1 thư mục đồng bộ

```bash
dvc remote add -d storage /absolute/path/to/shared/dvcstore
```

**Ưu điểm**
- ít setup nhất
- dễ hiểu
- hợp với project nhỏ

**Nhược điểm**
- không đẹp cho nhóm làm việc phân tán trên nhiều máy khác nhau

---

### Lựa chọn 2 — **Google Drive remote** (thực tế hơn cho nhóm sinh viên phân tán)
Phù hợp nếu:
- nhóm muốn mỗi người có thể `dvc pull` / `dvc push` từ xa
- chưa có S3 / GCS / Azure

```bash
dvc remote add -d teamdrive gdrive://<folder-id>
```

Nếu cần custom credential:

```bash
dvc remote modify teamdrive gdrive_client_id "<client-id>"
dvc remote modify teamdrive gdrive_client_secret "<client-secret>"
```

Thông tin nhạy cảm nên để local:

```bash
dvc remote modify --local teamdrive gdrive_client_id "<client-id>"
dvc remote modify --local teamdrive gdrive_client_secret "<client-secret>"
```

> Khuyến nghị thực dụng:  
> - Nếu nhóm chỉ cần mức tối thiểu: dùng **shared folder remote**  
> - Nếu nhóm làm việc phân tán nhiều máy: dùng **Google Drive remote**  
> - Nếu Google Drive auth rườm rà, hãy giữ local remote trước rồi nâng cấp sau

## Push / Pull cơ bản

Sau khi đã `dvc add` artifact:

```bash
dvc push
git push
```

Máy khác trong nhóm:

```bash
git pull
dvc pull
```

### Ghi nhớ
- `git push` để đẩy **code + `.dvc` metadata**
- `dvc push` để đẩy **data/model artifact thật**
- Cả hai đều cần thiết

## Nếu sau này muốn migrate luôn raw dataset sang DVC

Đây là **bước tùy chọn**, không bắt buộc ngay bây giờ.

### Khi nào nên làm?
- raw dataset bắt đầu có nhiều phiên bản
- bạn muốn repo gọn hơn
- bạn không muốn tiếp tục giữ raw data lớn trong Git

### Cách làm tối thiểu
Ví dụ bạn muốn chuyển `raw_data/` sang `data/raw/`:

```bash
mkdir -p data
mv raw_data data/raw
dvc add data/raw
git add data/raw.dvc .gitignore
git commit -m "Move raw dataset under DVC tracking"
```

### Lưu ý quan trọng
- Notebook này **không** đề xuất rewrite toàn bộ Git history ở giai đoạn hiện tại
- Với project sinh viên, giữ snapshot cũ như hiện tại và **ngừng đưa thêm artifact lớn vào Git** thường đã đủ thực tế

## Điều kiện nào khiến sau này nên bắt đầu tích hợp DVC ngay?

Nếu một trong các điều kiện sau xuất hiện, hãy bắt đầu DVC ngay:
1. Bắt đầu sinh ra nhiều file `interim` / `processed`
2. Bắt đầu lưu nhiều model `.pkl`, `.joblib`, `.bin`
3. Nhiều thành viên nhóm cần đồng bộ data / model
4. Kết quả thí nghiệm bắt đầu khó tái lập
5. Repo bắt đầu nặng vì artifact lớn thay đổi thường xuyên
6. Bạn muốn tiến thêm một bước theo hướng MLOps thay vì chỉ làm notebook đơn lẻ

# End of Notebook

Notebook này đã hoàn thành 3 mục tiêu bạn yêu cầu:
1. **EDA thực chiến cho fraud detection**
2. **Đánh giá có nên dùng DVC hay không**
3. **Đưa ra cách triển khai DVC tối thiểu, đủ dùng cho project sinh viên theo hướng MLOps**